# 08 - Simulação Excluindo Slots Vermelhos

Estratégias **1-2-4** e **1-2** operando em todos os horários **exceto os vermelhos**.
- Vermelho = qualquer tom de vermelho/laranja no heatmap de % LOWs
- Banca inicial: R$500,00
- Saque de R$100 ao atingir R$600
- Reposição de R$500 em caso de banca zerada

In [ ]:
import sys
sys.path.insert(0, '.')
from config_analysis import *

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import plotly.io as pio
pio.templates.default = 'plotly_dark'

CYAN, MAGENTA, GREEN, RED, YELLOW = '#00f0ff', '#ff00ff', '#00ff88', '#ff3366', '#ffff00'

df = load_processed_data()
if df is None:
    raise FileNotFoundError('Execute o notebook 01 primeiro!')

print(f'Dataset: {len(df):,} registros')
print(f'Período: {df["date"].min().strftime("%Y-%m-%d")} a {df["date"].max().strftime("%Y-%m-%d")}')

## 1. Identificar Slots Vermelhos no Heatmap

In [ ]:
# Montar heatmap Hora x Dia (mesmo do notebook 02)
heatmap = df.groupby(['dia_semana', 'hora'])['is_low'].mean() * 100

# O heatmap usa escala RdYlGn_r:
#   min (verde escuro) → meio (amarelo) → max (vermelho escuro)
# O ponto de virada amarelo→vermelho é o MEIO da escala visual
data_min = heatmap.min()
data_max = heatmap.max()
midpoint = (data_min + data_max) / 2

print(f'Heatmap % LOW:')
print(f'  Mínimo:  {data_min:.2f}% (verde escuro)')
print(f'  Máximo:  {data_max:.2f}% (vermelho escuro)')
print(f'  Meio:    {midpoint:.2f}% (amarelo → acima disso é vermelho)')
print(f'  Média:   {heatmap.mean():.2f}%')

# Slots VERMELHOS = acima do ponto médio da escala visual
RED_SLOTS = set()
ALLOWED_SLOTS = set()
for (dow, hour), pct in heatmap.items():
    if pct > midpoint:
        RED_SLOTS.add((dow, hour))
    else:
        ALLOWED_SLOTS.add((dow, hour))

print(f'\nSlots vermelhos (excluídos): {len(RED_SLOTS)} de 168')
print(f'Slots permitidos (verde+amarelo): {len(ALLOWED_SLOTS)} de 168 ({len(ALLOWED_SLOTS)/168*100:.0f}%)')

In [ ]:
# Visualizar: quais slots foram excluídos
dias = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
dias_full = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Montar matriz para heatmap com marcação
heatmap_pivot = heatmap.unstack(fill_value=0)
heatmap_pivot.index = [dias[i] for i in heatmap_pivot.index]

# Texto: ✓ se permitido, ✗ se excluído
text_matrix = []
for d in range(7):
    row = []
    for h in range(24):
        pct = heatmap.get((d, h), 0)
        marker = '✓' if (d, h) in ALLOWED_SLOTS else '✗'
        row.append(f'{pct:.1f}%\n{marker}')
    text_matrix.append(row)

fig = go.Figure(go.Heatmap(
    z=heatmap_pivot.values,
    x=[str(h) for h in range(24)],
    y=[dias[i] for i in range(7)],
    colorscale='RdYlGn_r',
    text=text_matrix,
    texttemplate='%{text}',
    textfont=dict(size=9),
    colorbar=dict(title='% LOW'),
))

fig.add_hline  # dummy
fig.update_layout(
    title=f'Slots Permitidos (✓) vs Excluídos (✗) — Corte: {midpoint:.2f}%',
    xaxis_title='Hora', yaxis_title='',
    height=400, width=1000,
)
fig.show()

# Tabela resumida por dia
print(f'\nSlots permitidos por dia:')
for d in range(7):
    allowed_hours = sorted([h for (dw, h) in ALLOWED_SLOTS if dw == d])
    print(f'  {dias[d]:>3}: {len(allowed_hours):2d}h → {allowed_hours}')

## 2. Configuração e Dados

In [ ]:
BANKROLL_INICIAL = 500.0
META_SAQUE = 100.0
TRIGGER = 6
TARGET = 2.0

STRATEGIES = {
    '1-2-4': {'pattern': [1, 2, 4], 'divisor': 7},
    '1-2':   {'pattern': [1, 2],    'divisor': 3},
}

mults = df['multiplicador'].values
is_low = (mults < TARGET).astype(int)
hours = df['hora'].values
weekdays = df['dia_semana'].values
dates = df['date'].values

n = len(mults)
streaks = np.zeros(n, dtype=int)
for i in range(1, n):
    if is_low[i] == 1:
        streaks[i] = streaks[i - 1] + 1

allowed_mask = np.array([(weekdays[i], hours[i]) in ALLOWED_SLOTS for i in range(n)])

print(f'Rounds totais:             {n:,}')
print(f'Rounds permitidos:         {allowed_mask.sum():,} ({allowed_mask.mean()*100:.1f}%)')
print(f'Rounds excluídos (verm.):  {(~allowed_mask).sum():,} ({(~allowed_mask).mean()*100:.1f}%)')
print(f'Sinais (streak>={TRIGGER}):  {(streaks >= TRIGGER).sum():,}')
print(f'Sinais permitidos:           {((streaks >= TRIGGER) & allowed_mask).sum():,}')

## 3. Motor de Simulação

In [ ]:
def simulate_no_reds(
    mults, streaks, hours, weekdays, dates, allowed_slots,
    pattern, divisor, trigger, target,
    bankroll_inicial, meta_saque,
):
    n = len(mults)
    bankroll = bankroll_inicial
    total_deposited = bankroll_inicial
    total_withdrawn = 0.0
    n_deposits = 1
    n_withdrawals = 0

    equity_curve = []
    bankroll_curve = []
    trades = []
    withdrawals_log = []
    deposits_log = []
    cycle_unit = None

    for i in range(n - 1):
        current_streak = streaks[i]

        # Excluir slots vermelhos
        if (weekdays[i], hours[i]) not in allowed_slots:
            continue

        if bankroll <= 0:
            bankroll = bankroll_inicial
            total_deposited += bankroll_inicial
            n_deposits += 1
            deposits_log.append({'index': i, 'date': dates[i], 'amount': bankroll_inicial})

        if current_streak < trigger:
            cycle_unit = None
            continue

        position = current_streak - trigger

        if position >= len(pattern):
            continue

        if position == 0:
            cycle_unit = bankroll / divisor

        if cycle_unit is None or cycle_unit <= 0:
            continue

        bet = cycle_unit * pattern[position]
        bet = min(bet, bankroll)

        if bet < 0.01:
            continue

        next_mult = mults[i + 1]

        if next_mult >= target:
            pnl = bet * (target - 1)
            hit = True
        else:
            pnl = -bet
            hit = False

        bankroll += pnl
        bankroll = round(bankroll, 2)

        net_equity = total_withdrawn + bankroll - total_deposited

        trades.append({
            'index': i,
            'date': dates[i],
            'hour': hours[i],
            'weekday': weekdays[i],
            'streak': current_streak,
            'position': position,
            'bet': round(bet, 2),
            'next_mult': next_mult,
            'hit': hit,
            'pnl': round(pnl, 2),
            'bankroll': bankroll,
            'net_equity': net_equity,
        })

        equity_curve.append(net_equity)
        bankroll_curve.append(bankroll)

        if bankroll >= bankroll_inicial + meta_saque:
            withdraw = bankroll - bankroll_inicial
            total_withdrawn += withdraw
            n_withdrawals += 1
            withdrawals_log.append({
                'index': i, 'date': dates[i],
                'amount': withdraw, 'total': total_withdrawn,
            })
            bankroll = bankroll_inicial

        if bankroll <= 0:
            bankroll = bankroll_inicial
            total_deposited += bankroll_inicial
            n_deposits += 1
            deposits_log.append({'index': i, 'date': dates[i], 'amount': bankroll_inicial})

    trades_df = pd.DataFrame(trades)
    wins = trades_df['hit'].sum() if len(trades_df) > 0 else 0
    total_trades = len(trades_df)

    return {
        'trades_df': trades_df,
        'equity_curve': equity_curve,
        'bankroll_curve': bankroll_curve,
        'total_trades': total_trades,
        'wins': wins,
        'losses': total_trades - wins,
        'win_rate': wins / total_trades if total_trades > 0 else 0,
        'final_bankroll': bankroll,
        'total_deposited': total_deposited,
        'total_withdrawn': total_withdrawn,
        'n_deposits': n_deposits,
        'n_withdrawals': n_withdrawals,
        'net_profit': total_withdrawn + bankroll - total_deposited,
        'withdrawals_log': pd.DataFrame(withdrawals_log),
        'deposits_log': pd.DataFrame(deposits_log),
    }

print('Motor pronto.')

## 4. Executar Simulações

In [ ]:
results = {}

for name, cfg in STRATEGIES.items():
    print(f'\nSimulando {name} (sem vermelhos)...')
    r = simulate_no_reds(
        mults=mults, streaks=streaks, hours=hours,
        weekdays=weekdays, dates=dates,
        allowed_slots=ALLOWED_SLOTS,
        pattern=cfg['pattern'], divisor=cfg['divisor'],
        trigger=TRIGGER, target=TARGET,
        bankroll_inicial=BANKROLL_INICIAL,
        meta_saque=META_SAQUE,
    )
    results[name] = r

    print(f'  Trades:        {r["total_trades"]:,}')
    print(f'  Win Rate:      {r["win_rate"]*100:.2f}%')
    print(f'  Saques:        {r["n_withdrawals"]:,} (R${r["total_withdrawn"]:,.2f})')
    print(f'  Depósitos:     {r["n_deposits"]:,} (R${r["total_deposited"]:,.2f})')
    print(f'  Banca final:   R${r["final_bankroll"]:,.2f}')
    print(f'  LUCRO LÍQUIDO: R${r["net_profit"]:+,.2f}')

## 5. Comparação

In [ ]:
r1 = results['1-2-4']
r2 = results['1-2']
days = (df['date'].max() - df['date'].min()).days
months = days / 30.44

print('=' * 70)
print('COMPARAÇÃO (SEM VERMELHOS): 1-2-4 vs 1-2')
print('=' * 70)
print(f'{"":>25} {"1-2-4":>20} {"1-2":>20}')
print('-' * 70)

rows = [
    ('Total de Trades', f'{r1["total_trades"]:,}', f'{r2["total_trades"]:,}'),
    ('Win Rate', f'{r1["win_rate"]*100:.2f}%', f'{r2["win_rate"]*100:.2f}%'),
    ('Wins / Losses', f'{r1["wins"]:,} / {r1["losses"]:,}', f'{r2["wins"]:,} / {r2["losses"]:,}'),
    ('', '', ''),
    ('Total Depositado', f'R${r1["total_deposited"]:,.2f}', f'R${r2["total_deposited"]:,.2f}'),
    ('Total Sacado', f'R${r1["total_withdrawn"]:,.2f}', f'R${r2["total_withdrawn"]:,.2f}'),
    ('Nº Saques', f'{r1["n_withdrawals"]:,}', f'{r2["n_withdrawals"]:,}'),
    ('Nº Depósitos', f'{r1["n_deposits"]:,}', f'{r2["n_deposits"]:,}'),
    ('Banca Final', f'R${r1["final_bankroll"]:,.2f}', f'R${r2["final_bankroll"]:,.2f}'),
    ('', '', ''),
    ('LUCRO LÍQUIDO', f'R${r1["net_profit"]:+,.2f}', f'R${r2["net_profit"]:+,.2f}'),
    ('Lucro/mês', f'R${r1["net_profit"]/months:+,.2f}', f'R${r2["net_profit"]/months:+,.2f}'),
    ('ROI', f'{r1["net_profit"]/r1["total_deposited"]*100:+.1f}%', f'{r2["net_profit"]/r2["total_deposited"]*100:+.1f}%'),
]

for label, v1, v2 in rows:
    print(f'{label:>25} {v1:>20} {v2:>20}')

print(f'\nPeríodo: {days:,} dias ({months:.0f} meses)')
print(f'Slots excluídos: {len(RED_SLOTS)} de 168 ({len(RED_SLOTS)/168*100:.0f}%)')

## 6. Curvas de Equity

In [ ]:
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Lucro Líquido Acumulado (sem vermelhos)', 'Saldo da Banca'),
)

for name, color in [('1-2-4', GREEN), ('1-2', CYAN)]:
    r = results[name]
    tdf = r['trades_df']
    if len(tdf) == 0:
        continue

    fig.add_trace(go.Scatter(
        x=tdf['date'], y=r['equity_curve'],
        mode='lines', name=f'{name} (lucro)',
        line=dict(color=color, width=1.5),
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=tdf['date'], y=r['bankroll_curve'],
        mode='lines', name=f'{name} (banca)',
        line=dict(color=color, width=1, dash='dot'),
    ), row=2, col=1)

fig.add_hline(y=0, line_dash='dash', line_color=RED, opacity=0.5, row=1, col=1)
fig.add_hline(y=BANKROLL_INICIAL, line_dash='dash', line_color=YELLOW, opacity=0.5, row=2, col=1,
              annotation_text='Banca base')

fig.update_layout(height=800, title_text='Simulação Sem Vermelhos: 1-2-4 vs 1-2')
fig.update_yaxes(title_text='R$', row=1, col=1)
fig.update_yaxes(title_text='R$', row=2, col=1)
fig.show()

## 7. Saques vs Depósitos

In [ ]:
for name in ['1-2-4', '1-2']:
    r = results[name]
    wl = r['withdrawals_log']
    dl = r['deposits_log']

    fig = go.Figure()

    if len(wl) > 0:
        wl['cumulative'] = wl['amount'].cumsum()
        fig.add_trace(go.Scatter(
            x=wl['date'], y=wl['cumulative'],
            mode='lines', name='Saques acumulados',
            line=dict(color=GREEN, width=2),
            fill='tozeroy', fillcolor='rgba(0,255,136,0.08)',
        ))

    if len(dl) > 0:
        dl['cumulative'] = dl['amount'].cumsum()
        fig.add_trace(go.Scatter(
            x=dl['date'], y=dl['cumulative'],
            mode='lines', name='Depósitos acumulados',
            line=dict(color=RED, width=2),
            fill='tozeroy', fillcolor='rgba(255,51,102,0.08)',
        ))

    fig.update_layout(
        title=f'Estratégia {name} (sem vermelhos): Saques vs Depósitos',
        xaxis_title='', yaxis_title='R$ Acumulado',
        height=450,
    )
    fig.show()

    print(f'\n--- {name} ---')
    if len(wl) > 0:
        print(f'Primeiro saque: {pd.Timestamp(wl.iloc[0]["date"]).strftime("%Y-%m-%d %H:%M")}')
        print(f'Último saque:   {pd.Timestamp(wl.iloc[-1]["date"]).strftime("%Y-%m-%d %H:%M")}')
        print(f'Frequência:     1 saque a cada {len(r["trades_df"])/len(wl):.0f} trades')
    if len(dl) > 0:
        print(f'Reposições:     {len(dl)} (1 a cada {len(r["trades_df"])/len(dl):.0f} trades)')

## 8. Análise Mensal

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('1-2-4 (sem verm.)', '1-2 (sem verm.)'))

for col, name in enumerate(['1-2-4', '1-2'], 1):
    r = results[name]
    tdf = r['trades_df'].copy()
    if len(tdf) == 0:
        continue

    tdf['ano_mes'] = pd.to_datetime(tdf['date']).dt.strftime('%Y-%m')
    monthly = tdf.groupby('ano_mes').agg(
        pnl=('pnl', 'sum'),
        trades=('pnl', 'count'),
        wr=('hit', 'mean'),
    ).reset_index()

    fig.add_trace(go.Bar(
        x=monthly['ano_mes'], y=monthly['pnl'],
        marker_color=[GREEN if p > 0 else RED for p in monthly['pnl']],
        name=name,
        text=[f'R${p:+,.0f}' for p in monthly['pnl']],
        textposition='outside',
        textfont=dict(size=8),
    ), row=1, col=col)

fig.update_layout(height=500, title_text='PnL Mensal (Sem Vermelhos)', showlegend=False)
fig.update_yaxes(title_text='R$')
fig.show()

for name in ['1-2-4', '1-2']:
    r = results[name]
    tdf = r['trades_df'].copy()
    if len(tdf) == 0:
        continue
    tdf['ano_mes'] = pd.to_datetime(tdf['date']).dt.strftime('%Y-%m')
    monthly = tdf.groupby('ano_mes').agg(
        pnl=('pnl', 'sum'),
        trades=('pnl', 'count'),
        wr=('hit', 'mean'),
    ).reset_index()

    pos = (monthly['pnl'] > 0).sum()
    print(f'\n--- {name}: {pos}/{len(monthly)} meses positivos ({pos/len(monthly)*100:.0f}%) ---')
    print(f'{"Mês":>10} {"PnL":>12} {"Trades":>8} {"WR":>8}')
    print('-' * 42)
    for _, row in monthly.iterrows():
        print(f'{row["ano_mes"]:>10} R${row["pnl"]:>+10,.2f} {int(row["trades"]):>8} {row["wr"]*100:>7.1f}%')

## 9. Resumo Final

In [ ]:
days = (df['date'].max() - df['date'].min()).days
months = days / 30.44

print('=' * 60)
print('RESULTADO FINAL - SEM SLOTS VERMELHOS')
print('=' * 60)
print(f'Filtro: excluídos {len(RED_SLOTS)} slots com % LOW > {midpoint:.2f}%')
print(f'Operando em {len(ALLOWED_SLOTS)}/168 slots ({len(ALLOWED_SLOTS)/168*100:.0f}%)')

for name in ['1-2-4', '1-2']:
    r = results[name]
    cfg = STRATEGIES[name]

    print(f'\n--- Estratégia {name} (unit = banca/{cfg["divisor"]}) ---')
    print(f'  Banca inicial:     R${BANKROLL_INICIAL:,.2f}')
    print(f'  Total depositado:  R${r["total_deposited"]:,.2f} ({r["n_deposits"]} depósitos)')
    print(f'  Total sacado:      R${r["total_withdrawn"]:,.2f} ({r["n_withdrawals"]} saques)')
    print(f'  Banca final:       R${r["final_bankroll"]:,.2f}')
    print(f'  ')
    print(f'  LUCRO LÍQUIDO:     R${r["net_profit"]:+,.2f}')
    print(f'  Lucro/mês:         R${r["net_profit"]/months:+,.2f}')
    print(f'  ROI:               {r["net_profit"]/r["total_deposited"]*100:+.1f}%')
    print(f'  ')
    print(f'  Trades:            {r["total_trades"]:,}')
    print(f'  Win Rate:          {r["win_rate"]*100:.2f}%')

better = '1-2-4' if results['1-2-4']['net_profit'] > results['1-2']['net_profit'] else '1-2'
diff = abs(results['1-2-4']['net_profit'] - results['1-2']['net_profit'])
print(f'\n{"="*60}')
print(f'VENCEDORA: Estratégia {better} (R${diff:,.2f} a mais)')
print(f'{"="*60}')